In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score

In [2]:
# ----------------------------
# 1 Load Dataset
# ----------------------------

data = load_diabetes()

X = data.data
y = (data.target > np.mean(data.target)).astype(int)

df = pd.DataFrame(X)
df["target"] = y

In [3]:
# ----------------------------
# 2 Normalize
# ----------------------------

scaler = MinMaxScaler()
df.iloc[:, :-1] = scaler.fit_transform(df.iloc[:, :-1])

In [4]:
# ----------------------------
# 3 Split Data Across Hospitals
# ----------------------------

num_clients = 3
clients = np.array_split(df.sample(frac=1), num_clients)

In [5]:
# ----------------------------
# 4 Initialize Global Model
# ----------------------------

num_features = X.shape[1]
global_W = np.random.randn(num_features)
global_b = 0

In [6]:
# ----------------------------
# 5 Local Training Function
# ----------------------------

def sigmoid(z):
    return 1/(1+np.exp(-z))


def local_train(data, W, b, lr=0.1, epochs=20):

    X = data.iloc[:,:-1].values
    y = data["target"].values

    W_local = W.copy()
    b_local = b

    for _ in range(epochs):

        pred = sigmoid(np.dot(X,W_local)+b_local)

        error = pred - y

        grad_W = np.dot(X.T,error)/len(y)
        grad_b = np.mean(error)

        W_local -= lr*grad_W
        b_local -= lr*grad_b

    return W_local,b_local,len(y)

In [7]:
# ----------------------------
# 6 Federated Averaging
# ----------------------------

def fedavg(updates):

    total = sum(size for _,_,size in updates)

    new_W = np.zeros_like(updates[0][0])
    new_b = 0

    for W,b,size in updates:

        weight = size/total

        new_W += weight*W
        new_b += weight*b

    return new_W,new_b

In [8]:
# ----------------------------
# 7 Federated Training
# ----------------------------

rounds = 10

for r in range(rounds):

    updates=[]

    for client in clients:

        W_local,b_local,size = local_train(client,global_W,global_b)

        updates.append((W_local,b_local,size))

    global_W,global_b = fedavg(updates)

    print("Round",r+1,"completed")

AttributeError: 'numpy.ndarray' object has no attribute 'iloc'

In [9]:
# ----------------------------
# 7 Federated Training
# ----------------------------

rounds = 10

for r in range(rounds):

    updates=[]

    for client in clients:

        W_local,b_local,size = local_train(client,global_W,global_b)

        updates.append((W_local,b_local,size))

    global_W,global_b = fedavg(updates)

    print("Round",r+1,"completed")

AttributeError: 'numpy.ndarray' object has no attribute 'iloc'